# Advanced Trust Game Training

This notebook connects to the deployed Trust Game OpenEnv Space, collects live demonstrations, fine-tunes a LoRA adapter with TRL + Unsloth, and evaluates random vs heuristic vs trained policies.

Use a Colab GPU runtime before running the training cell.


In [ ]:
# Colab/GPU setup
%pip install -q --retries 10 --timeout 120 unsloth trl transformers datasets accelerate peft bitsandbytes matplotlib requests


In [ ]:
from __future__ import annotations

import argparse
import csv
import json
import math
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import requests


ACTION_KEYS = (
    "agent_id",
    "claim_amount",
    "verify_targets",
    "accept_proposal",
    "communication_act",
)
DEFAULT_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
SEED = 42

random.seed(SEED)


In [ ]:
# Configuration
BASE_URL = "https://hardikshreyas-trust-game-env.hf.space"
OUT_DIR = Path("eval_results/advanced_training")
DATASET_EPISODES = 80
EVAL_EPISODES = 24
MAX_STEPS = 32
REQUEST_DELAY_S = 0.35
MAX_SEQ_LENGTH = 1024
EPOCHS = 1.0
LEARNING_RATE = 2e-4
RUN_TRAINING = True

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Using Space:", BASE_URL)
print("Artifacts:", OUT_DIR)


In [ ]:
@dataclass
class EpisodeMetrics:
    policy: str
    episode: int
    total_reward: float
    steps: int
    deception_rate: float
    deception_detection_rate: float
    deception_effectiveness_score: float
    trust_stability: float
    trust_network_stability: float
    information_diffusion_rate: float
    betrayal_recognition_rate: float
    message_inconsistency_rate: float
    social_penalty_rate: float
    blackmail_signal_rate: float
    fairness: float
    efficiency: float
    long_term_payoff: float
    successful_deceptions: float


class TrustGameRemoteClient:
    """Small resilient HTTP client for the OpenEnv Trust Game Space."""

    def __init__(
        self,
        base_url: str,
        request_delay_s: float = 0.35,
        max_backoff_s: float = 12.0,
        timeout_s: float = 60.0,
    ) -> None:
        self.base_url = base_url.rstrip("/")
        self.request_delay_s = request_delay_s
        self.max_backoff_s = max_backoff_s
        self.timeout_s = timeout_s
        self.session = requests.Session()

    def post(
        self,
        path: str,
        payload: Optional[Dict[str, Any]] = None,
        retries: int = 5,
    ) -> Optional[requests.Response]:
        backoff = 1.0
        for attempt in range(retries + 1):
            try:
                if self.request_delay_s > 0:
                    time.sleep(self.request_delay_s)
                response = self.session.post(
                    f"{self.base_url}{path}",
                    json=payload,
                    timeout=self.timeout_s,
                )
            except Exception as exc:
                print(f"{path} request error {attempt + 1}/{retries + 1}: {exc}")
                time.sleep(min(backoff, self.max_backoff_s))
                backoff *= 2
                continue

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                wait_s = float(retry_after) if retry_after else min(backoff, self.max_backoff_s)
                print(f"{path} rate limited; sleeping {wait_s:.1f}s")
                time.sleep(wait_s)
                backoff *= 2
                continue

            if response.status_code >= 500:
                print(f"{path} server error {response.status_code}; retrying")
                print(response.text[:500])
                time.sleep(min(backoff, self.max_backoff_s))
                backoff *= 2
                continue

            return response

        return None

    def reset(self) -> Optional[Dict[str, Any]]:
        response = self.post("/reset", payload={})
        if response is None or response.status_code >= 400:
            return None
        return response.json()

    def step(self, action: Dict[str, Any], retries: int = 2) -> Optional[Dict[str, Any]]:
        payload = {"action": sanitize_action(action)}
        for attempt in range(retries + 1):
            response = self.post("/step", payload=payload, retries=3)
            if response is not None and response.status_code < 400:
                return response.json()
            print(f"/step failed {attempt + 1}/{retries + 1}; resetting session")
            self.reset()
        return None


In [ ]:
def clamp(value: float, low: float = 0.0, high: float = 100.0) -> float:
    return float(max(low, min(high, value)))

def sanitize_action(action: Dict[str, Any], fallback_agent_id: int = 0) -> Dict[str, Any]:
    raw_targets = action.get("verify_targets", [])
    if not isinstance(raw_targets, list):
        raw_targets = []

    communication_act = str(action.get("communication_act", "reassure")).strip().lower()
    allowed_acts = {"reassure", "threaten", "guilt", "false_promise", "partial_confession"}
    if communication_act not in allowed_acts:
        communication_act = "reassure"

    return {
        "agent_id": int(action.get("agent_id", fallback_agent_id)),
        "claim_amount": clamp(float(action.get("claim_amount", 40.0))),
        "verify_targets": [int(target) for target in raw_targets[:3]],
        "accept_proposal": bool(action.get("accept_proposal", False)),
        "communication_act": communication_act,
        "message": str(action.get("message", "trained-policy"))[:256],
    }


def safe_action_from_obs(obs: Dict[str, Any]) -> Dict[str, Any]:
    return sanitize_action(
        {
            "agent_id": obs.get("your_agent_id", 0),
            "claim_amount": obs.get("your_true_need", 40.0),
            "verify_targets": [],
            "accept_proposal": False,
            "communication_act": "reassure",
            "message": "safe fallback",
        }
    )

def sorted_low_trust_targets(obs: Dict[str, Any], k: int = 2) -> List[int]:
    agent_id = int(obs.get("your_agent_id", 0))
    trust_scores = obs.get("trust_scores", {}) or {}
    pairs = []
    for raw_id, raw_score in trust_scores.items():
        other_id = int(raw_id)
        if other_id != agent_id:
            pairs.append((other_id, float(raw_score)))
    return [agent_id for agent_id, _ in sorted(pairs, key=lambda pair: pair[1])[:k]]

def local_suspicion_scores(obs: Dict[str, Any]) -> Dict[int, float]:
    """Estimate suspicion from observable history, trust, and claim instability."""

    agent_id = int(obs.get("your_agent_id", 0))
    trust_scores = {int(k): float(v) for k, v in (obs.get("trust_scores", {}) or {}).items()}
    claim_history = obs.get("claim_history", {}) or {}
    all_claims = {int(k): float(v) for k, v in (obs.get("all_claims", {}) or {}).items()}

    scores: Dict[int, float] = {}
    for target_id, current_claim in all_claims.items():
        if target_id == agent_id:
            continue
        history = [float(x) for x in claim_history.get(str(target_id), claim_history.get(target_id, []))]
        if history:
            mean_claim = sum(history) / len(history)
            variance = sum((claim - mean_claim) ** 2 for claim in history) / len(history)
            inconsistency = min(1.0, math.sqrt(variance) / 30.0)
            deviation = min(1.0, abs(current_claim - mean_claim) / 40.0)
        else:
            inconsistency = 0.0
            deviation = 0.0
        low_trust = 1.0 - trust_scores.get(target_id, 0.5)
        scores[target_id] = clamp(0.45 * deviation + 0.35 * inconsistency + 0.20 * low_trust, 0.0, 1.0)

    return scores

def heuristic_action(obs: Dict[str, Any]) -> Dict[str, Any]:
    """Role-aware teacher policy with suspicion-based verification."""

    role = obs.get("your_role", "honest")
    agent_id = int(obs.get("your_agent_id", 0))
    need = float(obs.get("your_true_need", 40.0))
    metrics = obs.get("system_metrics", {}) or {}
    suspicious = [
        target_id
        for target_id, score in sorted(
            local_suspicion_scores(obs).items(),
            key=lambda pair: pair[1],
            reverse=True,
        )
        if score >= 0.45
    ]

    claim = need
    verify_targets: List[int] = []
    accept = False
    message = "claiming true need"
    communication_act = "reassure"

    if role == "honest":
        verify_targets = (suspicious or sorted_low_trust_targets(obs, k=2))[:2]
        accept = float(metrics.get("fairness", 0.0)) >= 0.7
        communication_act = "reassure"
    elif role == "self_interested":
        claim = min(100.0, need + 8.0)
        verify_targets = suspicious[:1]
        accept = True
        message = "prioritising my allocation while avoiding obvious overclaim"
        communication_act = "false_promise"
    elif role == "adversarial":
        claim = min(100.0, need + 18.0)
        accept = True
        message = "framing an inflated claim as reasonable need"
        communication_act = "guilt"
    elif role == "victim":
        verify_targets = suspicious[:2]
        accept = float(metrics.get("deception_rate", 0.0)) < 0.5
        message = "cautious trust with verification"
        communication_act = "reassure"
    elif role == "oversight":
        verify_targets = (suspicious or sorted_low_trust_targets(obs, k=2))[:2]
        accept = float(metrics.get("deception_rate", 1.0)) < 0.15
        message = "monitoring suspicious claims before approval"
        communication_act = "partial_confession"

    return sanitize_action(
        {
            "agent_id": agent_id,
            "claim_amount": claim,
            "verify_targets": verify_targets,
            "accept_proposal": accept,
            "communication_act": communication_act,
            "message": message,
        },
        fallback_agent_id=agent_id,
    )


def format_training_text(prompt: str, response: Dict[str, Any]) -> str:
    action_json = json.dumps({key: response[key] for key in ACTION_KEYS}, sort_keys=True)
    return (
        "You are an agent in the Trust Game environment.\n"
        "Return ONLY valid JSON with keys: agent_id, claim_amount, verify_targets, accept_proposal, communication_act.\n"
        "communication_act must be one of: reassure, threaten, guilt, false_promise, partial_confession.\n"
        "Use the role, trust scores, claims, and system metrics to choose the next action.\n\n"
        f"Prompt:\n{prompt}\n\n"
        f"Action JSON:\n{action_json}"
    )


In [ ]:
def collect_dataset(
    client: TrustGameRemoteClient,
    out_dir: Path,
    n_episodes: int,
    max_steps: int,
    seed: int,
) -> Dict[str, Any]:
    rows: List[Dict[str, Any]] = []
    episode_summaries: List[Dict[str, Any]] = []
    skipped = 0

    random.seed(seed)
    for episode in range(n_episodes):
        payload = client.reset()
        if payload is None or "observation" not in payload:
            skipped += 1
            continue

        obs = payload["observation"]
        done = bool(payload.get("done", False))
        total_reward = 0.0
        steps = 0
        while not done and steps < max_steps:
            action = heuristic_action(obs)
            rows.append(
                {
                    "episode": episode,
                    "step": steps,
                    "role": obs.get("your_role", "unknown"),
                    "prompt": obs.get("prompt", ""),
                    "response": json.dumps({key: action[key] for key in ACTION_KEYS}, sort_keys=True),
                    "text": format_training_text(obs.get("prompt", ""), action),
                }
            )

            next_payload = client.step(action)
            if next_payload is None:
                next_payload = client.step(safe_action_from_obs(obs))
            if next_payload is None or "observation" not in next_payload:
                skipped += 1
                break

            total_reward += float(next_payload.get("reward") or 0.0)
            obs = next_payload["observation"]
            done = bool(next_payload.get("done", False))
            steps += 1

        episode_summaries.append({"episode": episode, "steps": steps, "total_reward": total_reward})

    out_dir.mkdir(parents=True, exist_ok=True)
    dataset_path = out_dir / "trust_game_sft_dataset.jsonl"
    with dataset_path.open("w") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")

    summary = {
        "rows": len(rows),
        "episodes_requested": n_episodes,
        "episodes_skipped_or_partial": skipped,
        "dataset_path": str(dataset_path),
        "episode_summaries": episode_summaries,
    }
    (out_dir / "dataset_summary.json").write_text(json.dumps(summary, indent=2))
    return summary

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]


In [ ]:
def train_sft(
    dataset_path: Path,
    model_name: str,
    output_dir: Path,
    max_seq_length: int,
    epochs: float,
    learning_rate: float,
    seed: int,
) -> Dict[str, Any]:
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    from unsloth import FastLanguageModel

    output_dir.mkdir(parents=True, exist_ok=True)
    rows = load_jsonl(dataset_path)
    dataset = Dataset.from_list([{"text": row["text"]} for row in rows]).train_test_split(
        test_size=0.1,
        seed=seed,
    )

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=seed,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        dataset_text_field="text",
        args=SFTConfig(
            output_dir=str(output_dir),
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            num_train_epochs=epochs,
            learning_rate=learning_rate,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=50,
            save_steps=100,
            bf16=True,
            report_to="none",
            seed=seed,
        ),
    )

    train_output = trainer.train()
    final_dir = output_dir / "final"
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    history_path = output_dir / "trainer_log_history.json"
    history_path.write_text(json.dumps(trainer.state.log_history, indent=2))
    return {
        "model": model,
        "tokenizer": tokenizer,
        "trainer": trainer,
        "train_output": str(train_output),
        "final_dir": str(final_dir),
        "history_path": str(history_path),
    }


In [ ]:
def extract_first_json_object(text: str) -> Optional[Dict[str, Any]]:
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return parsed if isinstance(parsed, dict) else None

def model_action(model: Any, tokenizer: Any, obs: Dict[str, Any]) -> Dict[str, Any]:
    prompt = (
        "Return ONLY valid JSON with keys: agent_id, claim_amount, verify_targets, accept_proposal, communication_act.\n"
        "communication_act must be one of: reassure, threaten, guilt, false_promise, partial_confession.\n\n"
        f"Prompt:\n{obs.get('prompt', '')}\n\nAction JSON:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    parsed = extract_first_json_object(text) or {}
    parsed["message"] = "trained model"
    if "communication_act" not in parsed:
        parsed["communication_act"] = "reassure"
    return sanitize_action(parsed, fallback_agent_id=int(obs.get("your_agent_id", 0)))


def random_action(obs: Dict[str, Any]) -> Dict[str, Any]:
    agent_id = int(obs.get("your_agent_id", 0))
    others = sorted_low_trust_targets(obs, k=1)
    acts = ["reassure", "threaten", "guilt", "false_promise", "partial_confession"]
    return sanitize_action(
        {
            "agent_id": agent_id,
            "claim_amount": random.uniform(0.0, 100.0),
            "verify_targets": others if random.random() < 0.35 else [],
            "accept_proposal": random.random() < 0.5,
            "communication_act": random.choice(acts),
            "message": "random baseline",
        },
        fallback_agent_id=agent_id,
    )

def get_policy_action(
    policy: str,
    obs: Dict[str, Any],
    model: Any = None,
    tokenizer: Any = None,
) -> Dict[str, Any]:
    if policy == "random":
        return random_action(obs)
    if policy == "heuristic":
        return heuristic_action(obs)
    if policy == "trained":
        if model is None or tokenizer is None:
            raise ValueError("trained policy requires model and tokenizer")
        return model_action(model, tokenizer, obs)
    raise ValueError(f"Unknown policy: {policy}")

def run_eval(
    client: TrustGameRemoteClient,
    policies: Iterable[str],
    n_episodes: int,
    max_steps: int,
    model: Any = None,
    tokenizer: Any = None,
) -> List[EpisodeMetrics]:
    results: List[EpisodeMetrics] = []
    for policy in policies:
        for episode in range(n_episodes):
            payload = client.reset()
            if payload is None or "observation" not in payload:
                continue

            obs = payload["observation"]
            done = bool(payload.get("done", False))
            total_reward = 0.0
            steps = 0
            metrics: Dict[str, float] = {}
            while not done and steps < max_steps:
                action = get_policy_action(policy, obs, model=model, tokenizer=tokenizer)
                payload = client.step(action)
                if payload is None or "observation" not in payload:
                    break
                total_reward += float(payload.get("reward") or 0.0)
                obs = payload["observation"]
                metrics = obs.get("system_metrics", {}) or {}
                done = bool(payload.get("done", False))
                steps += 1

            results.append(
                EpisodeMetrics(
                    policy=policy,
                    episode=episode,
                    total_reward=total_reward,
                    steps=steps,
                    deception_rate=float(metrics.get("deception_rate", 0.0)),
                    deception_detection_rate=float(metrics.get("deception_detection_rate", 0.0)),
                    deception_effectiveness_score=float(metrics.get("deception_effectiveness_score", 0.0)),
                    trust_stability=float(metrics.get("trust_stability", 0.0)),
                    trust_network_stability=float(metrics.get("trust_network_stability", 0.0)),
                    information_diffusion_rate=float(metrics.get("information_diffusion_rate", 0.0)),
                    betrayal_recognition_rate=float(metrics.get("betrayal_recognition_rate", 0.0)),
                    message_inconsistency_rate=float(metrics.get("message_inconsistency_rate", 0.0)),
                    social_penalty_rate=float(metrics.get("social_penalty_rate", 0.0)),
                    blackmail_signal_rate=float(metrics.get("blackmail_signal_rate", 0.0)),
                    fairness=float(metrics.get("fairness", 0.0)),
                    efficiency=float(metrics.get("efficiency", 0.0)),
                    long_term_payoff=float(metrics.get("long_term_payoff", 0.0)),
                    successful_deceptions=float(metrics.get("successful_deceptions", 0.0)),
                )
            )
    return results

def mean(values: Iterable[float]) -> float:
    values = list(values)
    return sum(values) / max(1, len(values))

def write_eval_artifacts(results: List[EpisodeMetrics], out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = [result.__dict__ for result in results]
    csv_path = out_dir / "advanced_eval_results.csv"
    with csv_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    by_policy: Dict[str, List[EpisodeMetrics]] = {}
    for result in results:
        by_policy.setdefault(result.policy, []).append(result)

    summary = {}
    for policy, policy_results in by_policy.items():
        summary[policy] = {
            "episodes": len(policy_results),
            "mean_reward": mean(r.total_reward for r in policy_results),
            "mean_steps": mean(r.steps for r in policy_results),
            "mean_deception_rate": mean(r.deception_rate for r in policy_results),
            "mean_detection_rate": mean(r.deception_detection_rate for r in policy_results),
            "mean_deception_effectiveness": mean(r.deception_effectiveness_score for r in policy_results),
            "mean_trust_stability": mean(r.trust_stability for r in policy_results),
            "mean_trust_network_stability": mean(r.trust_network_stability for r in policy_results),
            "mean_information_diffusion_rate": mean(r.information_diffusion_rate for r in policy_results),
            "mean_betrayal_recognition_rate": mean(r.betrayal_recognition_rate for r in policy_results),
            "mean_message_inconsistency_rate": mean(r.message_inconsistency_rate for r in policy_results),
            "mean_social_penalty_rate": mean(r.social_penalty_rate for r in policy_results),
            "mean_blackmail_signal_rate": mean(r.blackmail_signal_rate for r in policy_results),
            "mean_fairness": mean(r.fairness for r in policy_results),
            "mean_long_term_payoff": mean(r.long_term_payoff for r in policy_results),
        }
    (out_dir / "advanced_eval_summary.json").write_text(json.dumps(summary, indent=2))

    try:
        import matplotlib.pyplot as plt

        plt.figure(figsize=(8, 4.5))
        for policy, policy_results in by_policy.items():
            plt.plot(
                [r.episode for r in policy_results],
                [r.total_reward for r in policy_results],
                marker="o",
                label=policy,
            )
        plt.xlabel("Evaluation episode")
        plt.ylabel("Total episode reward")
        plt.title("Trust Game: policy comparison")
        plt.legend()
        plt.tight_layout()
        plt.savefig(out_dir / "advanced_reward_comparison.png", dpi=180)

        plt.figure(figsize=(8, 4.5))
        labels = list(by_policy)
        rewards = [summary[label]["mean_reward"] for label in labels]
        plt.bar(labels, rewards)
        plt.xlabel("Policy")
        plt.ylabel("Mean total reward")
        plt.title("Trust Game: mean reward by policy")
        plt.tight_layout()
        plt.savefig(out_dir / "advanced_mean_reward.png", dpi=180)
    except Exception as exc:
        print(f"Skipping plots because matplotlib failed: {exc}")


In [ ]:
# Quick connectivity check
client = TrustGameRemoteClient(BASE_URL, request_delay_s=REQUEST_DELAY_S)
sample = client.reset()
print("Reset ok:", sample is not None)
if sample and "observation" in sample:
    print("Observation keys:", sorted(sample["observation"].keys()))
    print(sample["observation"].get("prompt", "")[:500])


In [ ]:
# Collect live demonstrations from the deployed environment
summary = collect_dataset(
    client=client,
    out_dir=OUT_DIR,
    n_episodes=DATASET_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED,
)
dataset_path = Path(summary["dataset_path"])
print(json.dumps({k: v for k, v in summary.items() if k != "episode_summaries"}, indent=2))


In [ ]:
# Train LoRA adapter with Unsloth + TRL
# Set RUN_TRAINING = False in the config cell if you only want dataset/eval artifacts.
model = None
tokenizer = None
policies = ["random", "heuristic"]

if RUN_TRAINING:
    training = train_sft(
        dataset_path=dataset_path,
        model_name=DEFAULT_MODEL,
        output_dir=OUT_DIR / "outputs" / "trust-game-sft-advanced",
        max_seq_length=MAX_SEQ_LENGTH,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        seed=SEED,
    )
    model = training["model"]
    tokenizer = training["tokenizer"]
    policies.append("trained")
    print(json.dumps({k: v for k, v in training.items() if k not in {"model", "tokenizer", "trainer"}}, indent=2))


In [ ]:
# Evaluate random, heuristic, and trained policies
results = run_eval(
    client=client,
    policies=policies,
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    model=model,
    tokenizer=tokenizer,
)
write_eval_artifacts(results, OUT_DIR)
print(f"Wrote evaluation artifacts to {OUT_DIR}")

summary_path = OUT_DIR / "advanced_eval_summary.json"
print(summary_path.read_text())


In [ ]:
# Display reward plots inside the notebook
from IPython.display import Image, display

for image_path in [OUT_DIR / "advanced_reward_comparison.png", OUT_DIR / "advanced_mean_reward.png"]:
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print("Missing plot:", image_path)
